In [ ]:
#1.Preparation of land use/land cover-based adaptive threshoding (LULC-AT) through GeoTIFF Export

import ee
import time

# === Initialize Earth Engine ===
ee.Initialize()

# === AOI: Extract Bangladesh boundary from FAO GAUL ===
gaul = ee.FeatureCollection("FAO/GAUL/2015/level0")
bangladesh = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh')).geometry()

# === Time period settings ===
year = 2021
month = 1
start_date = f"{year}-{month:02d}-01"
end_date = f"{year}-{month+1:02d}-01" if month < 12 else f"{year+1}-01-01"

print(f"Period: {start_date} - {end_date}")

# === Sentinel-2 NDWI2 ===
def mask_clouds(image):
    qa = image.select('QA60')
    opaque_clouds = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus_clouds = qa.bitwiseAnd(1 << 11).eq(0)
    mask = opaque_clouds.And(cirrus_clouds)
    return image.updateMask(mask)

def add_ndwi2(image):
    image = image.multiply(0.0001)
    ndwi2 = image.normalizedDifference(['B3', 'B8']).rename('NDWI2')
    return image.addBands(ndwi2)

s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(bangladesh)
    .filterDate(start_date, end_date)
    .map(mask_clouds)
    .map(add_ndwi2)
    .mean()
    .select('NDWI2')
)

# === Sentinel-1 VV/VH ===
def focal_filter(image):
    vv = image.select('VV').focal_median(1.5, 'circle', 'meters')
    vh = image.select('VH').focal_median(1.5, 'circle', 'meters')
    return ee.Image.cat([vv, vh]).rename(['VV', 'VH'])

s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(bangladesh)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .map(focal_filter)
    .median()
    .select(['VV', 'VH'])
)

# === ESA WorldCover ===
worldcover = (
    ee.ImageCollection('ESA/WorldCover/v200')
    .filterDate('2021-01-01', '2022-01-01')
    .first()
    .select('Map')
    .clip(bangladesh)
)

# === Label creation ===
label = s2.gt(0).rename('label')

# === Convert all bands to float32 ===
s1 = s1.toFloat()
s2 = s2.toFloat()
label = label.toFloat()
worldcover = worldcover.toFloat()

combined_image = s1.addBands([s2, label, worldcover.rename('landcover')])

# === Bounding box of Bangladesh ===
bounds = bangladesh.bounds().coordinates().getInfo()[0]

# Get longitude/latitude ranges
lons = [coord[0] for coord in bounds]
lats = [coord[1] for coord in bounds]

lon_min = min(lons)
lon_max = max(lons)
lat_min = min(lats)
lat_max = max(lats)

print(f"Bounding Box: lon {lon_min} ~ {lon_max}, lat {lat_min} ~ {lat_max}")

# === Split into 1° x 1° tiles and export ===
scale = 10
max_pixels = 1e13

tasks = []

for lon in range(int(lon_min), int(lon_max)):
    for lat in range(int(lat_min), int(lat_max)):
        tile = ee.Geometry.BBox(lon, lat, lon + 1, lat + 1).intersection(bangladesh, ee.ErrorMargin(100))

        task = ee.batch.Export.image.toDrive(
            image=combined_image.clip(tile),
            description=f'NDWI2_S1_WorldCover_{year}_{month:02d}_{lon}_{lat}',
            folder='GEE_Exports_NDWI_S1_LULC',
            region=tile,
            scale=scale,
            maxPixels=max_pixels,
            fileFormat='GeoTIFF'
        )
        task.start()
        tasks.append(task)
        print(f"Started export for tile: lon {lon} ~ {lon+1}, lat {lat} ~ {lat+1}")

print(f"Total {len(tasks)} export tasks have been started. Please check them in the Tasks tab.")


In [ ]:
#2.Identification of optimal VV thresholds for each LULC type. Code for other 5 formalae is included at the bottom.

import rasterio
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import glob
from tqdm import tqdm
import time
from datetime import datetime
import optuna
from optuna.samplers import TPESampler

# === Settings ===
year = 2021
month = 1
tif_pattern = f'/your/local/directory/{year}_{month:02d}/NDWI2_S1_WorldCover_{year}_{month:02d}_*.tif'
files = sorted(glob.glob(tif_pattern))
print(f"Number of tiles: {len(files)}")

# === Threshold range (for VV) ===
vv_th_min, vv_th_max = -22.0, -6.0

# === LULC descriptions (only necessary classes) ===
lc_descriptions = {
    10: 'Tree cover',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen'
}

# === Group definitions (20 and 60 combined) ===
# codes: list of class IDs included in the group
# label: identifier string for the Landcover column in output
# desc : description for the Description column
groups = [
    {"codes": [10],  "label": "10",     "desc": lc_descriptions[10]},
    {"codes": [20],  "label": "20",     "desc": lc_descriptions[20]},
    {"codes": [30],  "label": "30",     "desc": lc_descriptions[30]},
    {"codes": [40],  "label": "40",     "desc": lc_descriptions[40]},
    {"codes": [50],  "label": "50",     "desc": lc_descriptions[50]},
    {"codes": [60],  "label": "60",     "desc": lc_descriptions[60]},
    {"codes": [70],  "label": "70",     "desc": lc_descriptions[70]},
    {"codes": [80],  "label": "80",     "desc": lc_descriptions[80]},
    {"codes": [90],  "label": "90",     "desc": lc_descriptions[90]},
    {"codes": [95],  "label": "95",     "desc": lc_descriptions[95]},
    {"codes": [100], "label": "100",    "desc": lc_descriptions[100]},
    # ← merged group (20 and 60)
    {"codes": [20, 60], "label": "20_60", "desc": "Shrubland + Bare / sparse vegetation"},
]

# === Start time ===
start_time = time.time()
print(f"Process started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# === Load all data ===
vv_all, label_all, lc_all = [], [], []

for tif_path in tqdm(files, desc='Loading all tiles'):
    with rasterio.open(tif_path) as src:
        data = src.read()

    vv = data[0].flatten()        # VV
    vh = data[1].flatten()        # VH
    label = data[3].flatten()     # Label (water=1 / non-water=0 expected)
    landcover = data[4].flatten() # ESA WorldCover

    valid = ~np.isnan(vv) & ~np.isnan(label) & ~np.isnan(landcover)
    vv_all.append(vv[valid])
    vh_all.append(vh[valid])
    label_all.append(label[valid].astype(int))
    lc_all.append(landcover[valid].astype(int))

vv = np.concatenate(vv_all)
vh = np.concatenate(vh_all)
label = np.concatenate(label_all)
lc = np.concatenate(lc_all)

print(f"Total number of pixels: {len(vv)}")

# === Optimization by group ===
results = []
total_tp = total_tn = total_fp = total_fn = total_samples = 0

for g in tqdm(groups, desc='Optimizing by group'):
    codes = g["codes"]
    mask = np.isin(lc, codes)
    if not np.any(mask):
        continue

    vv_g = vv[mask]
    label_g = label[mask]

    def objective(trial):
        th = trial.suggest_float("vv_th", vv_th_min, vv_th_max, step=0.1)
        pred = (vv_g < th).astype(int)  # lower values are classified as water
        tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
        prec = tp / (tp + fp) if (tp + fp) else 0
        rec  = tp / (tp + fn) if (tp + fn) else 0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        return f1

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=100, show_progress_bar=False)

    best_th = study.best_params['vv_th']

    # Evaluate with the best threshold
    pred = (vv_g < best_th).astype(int)
    tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
    acc  = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec  = tp / (tp + fn) if (tp + fn) else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

    # Cumulative totals (for overall metrics; exclude 20,60 but include merged 20_60)
    if g["label"] not in ["20", "60"]:
        total_tp += tp
        total_tn += tn
        total_fp += fp
        total_fn += fn
        total_samples += len(vv_g)
    
    results.append({
        'Level': 'Subtotal',
        'Landcover': g["label"],          # e.g., '20_60'
        'Description': g["desc"],         # e.g., 'Shrubland + Bare / sparse vegetation'
        'Best_Threshold': round(best_th, 1),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1,
        'Samples': len(vv_g)
    })

# === Overall metrics (sum including merged group) ===
overall_acc = (total_tp + total_tn) / (total_tp + total_tn + total_fp + total_fn) if (total_tp + total_tn + total_fp + total_fn) else 0
overall_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
overall_rec  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
overall_f1   = 2 * overall_prec * overall_rec / (overall_prec + overall_rec) if (overall_prec + overall_rec) else 0

results.append({
    'Level': 'Overall',
    'Landcover': 'Overall',
    'Description': 'All classes total (with 20+60 merged)',
    'Best_Threshold': '-',
    'TP': total_tp, 'TN': total_tn, 'FP': total_fp, 'FN': total_fn,
    'Accuracy': overall_acc,
    'Precision': overall_prec,
    'Recall': overall_rec,
    'F1_Score': overall_f1,
    'Samples': total_samples
})

# === Export results to CSV ===
results_df = pd.DataFrame(results)
output_path = f'VV_{year}_{month:02d}.csv'
results_df.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
end_time = time.time()
elapsed_min = (end_time - start_time) / 60
print(f"Process finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total processing time: {elapsed_min:.2f} minutes")


##### 2.VH #####　2.VH #####　2.VH #####　2.VH #####　2.VH #####　
# vh_th_min, vh_th_max = -28.0, -15.0

# for g in tqdm(groups, desc='Optimizing by group'):
#     codes = g["codes"]
#     mask = np.isin(lc, codes)
#     if not np.any(mask):
#         continue

#     vh_g = vh[mask]
#     label_g = label[mask]

#     def objective(trial):
#         th = trial.suggest_float("vh_th", vh_th_min, vh_th_max, step=0.1)
#         pred = (vh_g < th).astype(int)
#         tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
#         prec = tp / (tp + fp) if (tp + fp) else 0
#         rec  = tp / (tp + fn) if (tp + fn) else 0
#         f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
#         return f1

#     sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction="maximize", sampler=sampler)
#     study.optimize(objective, n_trials=100, show_progress_bar=False)

#     best_th = study.best_params['vh_th']
##### 2.VH #####　2.VH #####　2.VH #####　2.VH #####　2.VH #####　

##### 3.(VV−VH)/(VV+VH) #####　3.(VV−VH)/(VV+VH) #####　3.(VV−VH)/(VV+VH) #####
# index_th_min, index_th_max = -1.0, 1.0

# # dB -> linear
# vv_lin = np.power(10.0, vv_db / 10.0)
# vh_lin = np.power(10.0, vh_db / 10.0)
# denom = vv_lin + vh_lin
# valid = np.isfinite(vv_lin) & np.isfinite(vh_lin) & np.isfinite(label) & np.isfinite(landcover) & (denom > 0)

# index = (vv - vh) / (vv + vh)

# for g in tqdm(groups, desc='Optimizing by group'):
#     codes = g["codes"]
#     mask = np.isin(lc, codes)
#     if not np.any(mask):
#         continue

#     idx_g = index[mask]
#     label_g = label[mask]

#     def objective(trial):
#         th = trial.suggest_float("index_th", index_th_min, index_th_max, step=0.1)
#         pred = (idx_g < th).astype(int)
#         tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
#         prec = tp / (tp + fp) if (tp + fp) else 0
#         rec  = tp / (tp + fn) if (tp + fn) else 0
#         f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
#         return f1

#     sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction="maximize", sampler=sampler)
#     study.optimize(objective, n_trials=21, show_progress_bar=False)

#     best_th = study.best_params['index_th']
##### 3.(VV−VH)/(VV+VH) #####　3.(VV−VH)/(VV+VH) #####　3.(VV−VH)/(VV+VH) #####

##### 4.4×VH/(VV+VH) #####　4.4×VH/(VV+VH) #####　4.4×VH/(VV+VH) #####
# index_th_min, index_th_max = 0.0, 4.0

# # dB -> linear
# vv_lin = np.power(10.0, vv_db / 10.0)
# vh_lin = np.power(10.0, vh_db / 10.0)
# denom = vv_lin + vh_lin
# valid = np.isfinite(vv_lin) & np.isfinite(vh_lin) & np.isfinite(label) & np.isfinite(landcover) & (denom > 0)

# index = 4.0 * vh / (vv + vh)

# for g in tqdm(groups, desc='Optimizing by group'):
#     codes = g["codes"]
#     mask = np.isin(lc, codes)
#     if not np.any(mask):
#         continue

#     idx_g = index[mask]
#     label_g = label[mask]

#     def objective(trial):
#         th = trial.suggest_float("index_th", index_th_min, index_th_max, step=0.1)
#         pred = (idx_g < th).astype(int)
#         tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
#         prec = tp / (tp + fp) if (tp + fp) else 0
#         rec  = tp / (tp + fn) if (tp + fn) else 0
#         f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
#         return f1

#     sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction="maximize", sampler=sampler)
#     study.optimize(objective, n_trials=41, show_progress_bar=False)

#     best_th = study.best_params['index_th']
##### 4.4×VH/(VV+VH) #####　4.4×VH/(VV+VH) #####　4.4×VH/(VV+VH) #####

##### 5.VV_VH_Intersection #####　5.VV_VH_Intersection #####　5.VV_VH_Intersection #####
# vv_th_min, vv_th_max = -22.0, -1.0
# vh_th_min, vh_th_max = -28.0, -1.0

# for g in tqdm(groups, desc='Optimizing by group'):
#     codes = g["codes"]
#     mask = np.isin(lc, codes)
#     if not np.any(mask):
#         continue

#     vv_g = vv[mask]
#     vh_g = vh[mask]
#     label_g = label[mask]

#     def objective(trial):
#         vv_th = trial.suggest_float("vv_th", vv_th_min, vv_th_max, step=0.1)
#         vh_th = trial.suggest_float("vh_th", vh_th_min, vh_th_max, step=0.1)
#         pred = ((vv_g < vv_th) & (vh_g < vh_th)).astype(int)
#         tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
#         prec = tp / (tp + fp) if (tp + fp) else 0
#         rec  = tp / (tp + fn) if (tp + fn) else 0
#         f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
#         return f1

#     sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction="maximize", sampler=sampler)
#     study.optimize(objective, n_trials=200, show_progress_bar=False)

#     best_vv_th = study.best_params['vv_th']
#     best_vh_th = study.best_params['vh_th']
##### 5.VV_VH_Intersection #####　5.VV_VH_Intersection #####　5.VV_VH_Intersection #####

##### 6.VV_VH_Union #####　6.VV_VH_Union #####　6.VV_VH_Uniion #####
# vv_th_min, vv_th_max = -24.0, -6.0
# vh_th_min, vh_th_max = -45.0, -15.0

# for g in tqdm(groups, desc='Optimizing by group'):
#     codes = g["codes"]
#     mask = np.isin(lc, codes)
#     if not np.any(mask):
#         continue

#     vv_g = vv[mask]
#     vh_g = vh[mask]
#     label_g = label[mask]

#     def objective(trial):
#         vv_th = trial.suggest_float("vv_th", vv_th_min, vv_th_max, step=0.1)
#         vh_th = trial.suggest_float("vh_th", vh_th_min, vh_th_max, step=0.1)
#         pred = ((vv_g < vv_th) | (vh_g < vh_th)).astype(int)
#         tn, fp, fn, tp = confusion_matrix(label_g, pred, labels=[0, 1]).ravel()
#         prec = tp / (tp + fp) if (tp + fp) else 0
#         rec  = tp / (tp + fn) if (tp + fn) else 0
#         f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
#         return f1

#     sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction="maximize", sampler=sampler)
#     study.optimize(objective, n_trials=300, show_progress_bar=False)

#     best_vv_th = study.best_params['vv_th']
#     best_vh_th = study.best_params['vh_th']
##### 6.VV_VH_Union #####　6.VV_VH_Union #####　6.VV_VH_Uniion #####

In [ ]:
#3.Visualization of surface water mapping with VV using LULC-AT

import geemap
import ee
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os
import matplotlib.patches as mpatches
from matplotlib.patheffects import withStroke
from shapely.geometry import shape
from cartopy.feature import ShapelyFeature

# Initialize Earth Engine
ee.Initialize()

# ========= Year/Month, Sensor, and Index =========
year = 2021
month = 1

satellite = 'Sentinel-1'        # 'Sentinel-1','Sentinel-2','Landsat-8'
selected_layer = 'VV'
# S1: 'VV','VH','VVVH_RATIO','FOURVH_OVER_SUM','VV_AND_VH', 'VV_OR_VH'
# S2/L8: 'NDWI','MNDWI','WI2015'

# ========= Analysis Region =========
REGION_KEY = 'country'
# Notes: country: 500 m (or 1000 m) / 100 km, Dhaka: 20 m / 10 km, Haor: 20 m / 10 km
REGION_KEYS = None

scale = 500  # meters
length_scalebar = 100  # km

# ============= National Boundary (Bangladesh) =============
gaul = ee.FeatureCollection("FAO/GAUL/2015/level0")
bangladesh = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh')).geometry()

def bbox(w, s, e, n): 
    return ee.Geometry.BBox(w, s, e, n)

REGION_CATALOG = {
    "Dhaka": bbox(90.441, 23.528, 90.741, 23.828),
    "Haor": bbox(90.982, 24.880, 91.282, 25.180),
}

if REGION_KEYS:
    geoms = [REGION_CATALOG[k] for k in REGION_KEYS]
    region_named = ee.Geometry.MultiGeometry(geoms).union()
    export_region = region_named.intersection(bangladesh, ee.ErrorMargin(1))
    region_label = "+".join(REGION_KEYS)
elif REGION_KEY == 'country':
    export_region = bangladesh
    region_label = "country"
else:
    if REGION_KEY not in REGION_CATALOG:
        raise ValueError(f"Unknown region name: {REGION_KEY}")
    export_region = REGION_CATALOG[REGION_KEY].intersection(bangladesh, ee.ErrorMargin(1))
    region_label = REGION_KEY

# ========= Date Range =========
start_date = f'{year}-{month:02d}-01'
end_date = f'{year+1}-01-01' if month == 12 else f'{year}-{month+1:02d}-01'

# ========= Optical (L8/S2) Preprocessing =========
def mask_clouds_L8(image):
    qa = image.select('QA_PIXEL')
    cloud  = (qa.rightShift(8).bitwiseAnd(3)).lte(1)
    shadow = (qa.rightShift(10).bitwiseAnd(3)).lte(1)
    snow   = (qa.rightShift(12).bitwiseAnd(3)).lte(1)
    cirrus = (qa.rightShift(14).bitwiseAnd(3)).lte(1)
    return image.updateMask(cloud.And(shadow).And(snow).And(cirrus))

def apply_scale_offset_L8(image):
    bands = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
    return image.addBands(image.select(bands).multiply(2.75e-05).add(-0.2), overwrite=True)

def calc_indices_L8(image):
    ndwi  = image.normalizedDifference(['SR_B3','SR_B5']).rename('NDWI')   # G vs NIR
    mndwi = image.normalizedDifference(['SR_B3','SR_B6']).rename('MNDWI')  # G vs SWIR1
    wi2015 = (image.select(['SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
              .multiply([171,3,-70,-45,-71]).reduce(ee.Reducer.sum())
              .add(1.7204).rename('WI2015'))
    return image.addBands([ndwi, mndwi, wi2015])

def mask_clouds_S2(image):
    qa = image.select('QA60')
    return image.updateMask(qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0))) \
                .select(['B3','B4','B8','B11','B12'])

def apply_scale_S2(image):
    return image.addBands(image.select(['B3','B4','B8','B11','B12']).multiply(0.0001), overwrite=True)

def calc_indices_S2(image):
    ndwi  = image.normalizedDifference(['B3','B8']).rename('NDWI')     # G vs NIR
    mndwi = image.normalizedDifference(['B3','B11']).rename('MNDWI')   # G vs SWIR1
    wi2015 = (image.select(['B3','B4','B8','B11','B12'])
              .multiply([171,3,-70,-45,-71]).reduce(ee.Reducer.sum())
              .add(1.7204).rename('WI2015'))
    return image.addBands([ndwi, mndwi, wi2015])

# ========= S1 Preprocessing and Metrics (create both dB and linear) =========
def s1_focal_filter(image):
    vv_db = image.select('VV').focal_median(1.5, 'circle', 'meters').rename('VV_DB')
    vh_db = image.select('VH').focal_median(1.5, 'circle', 'meters').rename('VH_DB')
    return ee.Image.cat([vv_db, vh_db])

def add_s1_metrics(image):
    vv_db = image.select('VV_DB')
    vh_db = image.select('VH_DB')

    # dB → linear : lin = 10^(dB/10)
    vv_lin = ee.Image(10.0).pow(vv_db.divide(10.0)).rename('VV_LIN')
    vh_lin = ee.Image(10.0).pow(vh_db.divide(10.0)).rename('VH_LIN')

    # Ratios on linear scale (avoid division by zero)
    sum_lin = vv_lin.add(vh_lin).max(1e-10)
    vvvh_ratio       = vv_lin.subtract(vh_lin).divide(sum_lin).rename('VVVH_RATIO')
    fourvh_over_sum  = vh_lin.multiply(4.0).divide(sum_lin).rename('FOURVH_OVER_SUM')

    return image.addBands([vv_lin, vh_lin, vvvh_ratio, fourvh_over_sum])

# ========= Build the Image =========
if satellite == 'Landsat-8':
    image = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
             .filterBounds(export_region).filterDate(start_date, end_date)
             .map(mask_clouds_L8).map(apply_scale_offset_L8).map(calc_indices_L8)
             .mean().clip(export_region))
elif satellite == 'Sentinel-2':
    image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(export_region).filterDate(start_date, end_date)
             .map(mask_clouds_S2).map(apply_scale_S2).map(calc_indices_S2)
             .mean().clip(export_region))
elif satellite == 'Sentinel-1':
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(export_region).filterDate(start_date, end_date)
          .filter(ee.Filter.eq('instrumentMode','IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
          .map(s1_focal_filter))
    image = s1.median().clip(export_region)
    image = add_s1_metrics(image)
else:
    raise ValueError("Satellite must be 'Sentinel-1', 'Sentinel-2', or 'Landsat-8'.")

# ========= Binary Visualization Utility =========
def visualize_index_binary(image, band, comparator='gt', thr=0, water_palette='#0000FF', non_palette='#f0f0f0'):
    if comparator == 'gt':
        water = image.select(band).gt(thr).selfMask()
        non   = image.select(band).lte(thr).selfMask()
    elif comparator == 'lt':
        water = image.select(band).lt(thr).selfMask()
        non   = image.select(band).gte(thr).selfMask()
    else:
        raise ValueError("comparator must be 'gt' or 'lt'")
    water_vis = water.visualize(min=0, max=1, palette=[water_palette])
    non_vis   = non.visualize(min=0, max=1, palette=[non_palette])
    return ee.ImageCollection([non_vis, water_vis]).mosaic(), water

# LULC (WorldCover)
worldcover = (
    ee.ImageCollection('ESA/WorldCover/v200')
    .filterDate('2021-01-01', '2022-01-01')
    .first()
    .select('Map')
    .clip(bangladesh)
)

# Three-class composite: water=blue, non-water=light gray, no-data=dark gray
def make_three_class_vis_from_masks(water_mask, valid_mask,
                                    water_col='#0000FF',
                                    non_col='#f5f5f5',  # #d9d9d9
                                    nodata_col='#666666'):
    # Convert to 0/1 binary before logical ops (important)
    w = water_mask.unmask(0)
    v = valid_mask.unmask(0)

    non    = v.And(w.eq(0)).selfMask()   # valid AND non-water
    nodata = v.Not().selfMask()          # invalid (no data)

    return ee.ImageCollection([
        nodata.visualize(min=0, max=1, palette=[nodata_col]),
        non.visualize(min=0, max=1, palette=[non_col]),
        w.selfMask().visualize(min=0, max=1, palette=[water_col]),
    ]).mosaic()

# ========= Load and Apply LULC-Specific Optimal Thresholds (S1 only) =========
import pandas as pd

# Mapping: metric -> folder name (under the base directory)
INDEX_FOLDER_MAP = {
    'VV': 'VV',
    'VH': 'VH',
    'VVVH_RATIO': 'VV_VH_Normalized',
    'FOURVH_OVER_SUM': 'VV_VH_Ratio4',
    'VV_AND_VH': 'VV_VH_Intersection',
    'VV_OR_VH': 'VV_VH_Union',
}

# Comparator for visualization
COMPARATOR_MAP = {
    'VV': ('VV_DB', 'lt'),
    'VH': ('VH_DB', 'lt'),
    'VVVH_RATIO': ('VVVH_RATIO', 'lt'),
    'FOURVH_OVER_SUM': ('FOURVH_OVER_SUM', 'gt'),
}

# Fallbacks (used if a threshold is missing in CSV)
FALLBACKS = {
    'VV': None,
    'VH': None,
    'VVVH_RATIO': None,
    'FOURVH_OVER_SUM': None,
    'VV_AND_VH': (None, None),
    'VV_OR_VH': (None, None),
}

# Required thresholds check (for 20 & 60, use the merged key '20_60' only)
def _assert_thresholds_complete(thr_map, selected_layer):
    required = [10, 30, 40, 50, 80, 90, 95, '20_60']
    missing = []
    if selected_layer in ('VV_AND_VH', 'VV_OR_VH'):
        # Need a pair (vv_thr, vh_thr)
        for k in required:
            v = thr_map.get(k)
            if (v is None) or (not isinstance(v, (list, tuple))) or (len(v) != 2):
                missing.append(k)
    else:
        # Single threshold
        for k in required:
            if thr_map.get(k) is None:
                missing.append(k)

    if missing:
        raise ValueError(f"Missing thresholds in CSV for: {missing}")

def _read_threshold_table(base_dir, selected_layer, year, month):
    """Read LULC-specific optimal thresholds from CSV.
       Returns:
       - Single-threshold types: {10:thr, 30:thr, ..., 95:thr, '20_60':thr}
       - AND/OR types: {10:(vv_thr, vh_thr), ..., '20_60':(vv_thr, vh_thr)}
    """
    folder = INDEX_FOLDER_MAP[selected_layer]
    fname  = f"{folder}_{year}_{month:02d}.csv"  # e.g., VV_2021_07.csv / VV_VH_Intersection_2021_07.csv
    # Use the provided base_dir (changed to '/your/local/directory' per request)
    csv_path = os.path.join(base_dir, folder, fname)

    # Auto-detect delimiter (comma/tab)
    df = pd.read_csv(csv_path, sep=None, engine="python")

    # Use only "Subtotal" rows
    df_sub = df[df["Level"].str.lower() == "subtotal"].copy()
    df_sub["Landcover"] = df_sub["Landcover"].astype(str)

    # Class codes (WorldCover v200). 20 & 60 will be merged into 20_60.
    cls_codes = ["10","30","40","50","80","90","95"]
    out = {}

    if selected_layer in ["VV_AND_VH", "VV_OR_VH"]:
        # Dual thresholds
        for code in cls_codes:
            r = df_sub[df_sub["Landcover"] == code]
            if len(r):
                out[int(code)] = (float(r["Best_VV_Threshold"].iloc[0]),
                                  float(r["Best_VH_Threshold"].iloc[0]))
        # 20_60
        r2060 = df_sub[df_sub["Landcover"].isin(["20_60"])]
        if len(r2060):
            out["20_60"] = (float(r2060["Best_VV_Threshold"].iloc[0]),
                            float(r2060["Best_VH_Threshold"].iloc[0]))
    else:
        # Single thresholds
        for code in cls_codes:
            r = df_sub[df_sub["Landcover"] == code]
            if len(r):
                out[int(code)] = float(r["Best_Threshold"].iloc[0])
        # 20_60
        r2060 = df_sub[df_sub["Landcover"].isin(["20_60"])]
        if len(r2060):
            out["20_60"] = float(r2060["Best_Threshold"].iloc[0])

    return out

def _apply_single_threshold_per_lulc(image, worldcover, band, comparator, thr_map, fallbacks):
    """Single-threshold types (VV, VH, VVVH_RATIO, FOURVH_OVER_SUM)"""
    wc = worldcover
    water = ee.Image(0)

    def cond_from_thr(thr):
        if thr is None:
            return ee.Image(0).eq(1)  # always False
        if comparator == 'lt':
            return image.select(band).lt(ee.Number(thr))
        else:
            return image.select(band).gt(ee.Number(thr))

    # Apply 20_60 (Shrubland ∪ Bare) first
    thr2060 = thr_map.get("20_60", fallbacks)
    c2060 = cond_from_thr(thr2060).int()
    m2060 = wc.eq(20).Or(wc.eq(60))
    water = water.where(m2060, c2060)

    # Remaining classes
    for code in [10, 30, 40, 50, 80, 90, 95]:
        thr = thr_map.get(code, fallbacks)
        c = cond_from_thr(thr).int()
        m = wc.eq(code)
        water = water.where(m, c)

    return water.selfMask()

def _apply_dual_threshold_per_lulc(image, worldcover, op, thr_map, fallbacks):
    """AND/OR (two thresholds for VV_DB and VH_DB)"""
    wc = worldcover
    water = ee.Image(0)

    def cond_pair(pair):
        vv_thr, vh_thr = pair
        vv_ok = image.select('VV_DB').lt(ee.Number(vv_thr)) if vv_thr is not None else ee.Image(0).eq(1)
        vh_ok = image.select('VH_DB').lt(ee.Number(vh_thr)) if vh_thr is not None else ee.Image(0).eq(1)
        return vv_ok.And(vh_ok) if op == 'AND' else vv_ok.Or(vh_ok)

    # 20_60 (Shrubland ∪ Bare)
    pair2060 = thr_map.get("20_60", fallbacks)
    c2060 = cond_pair(pair2060).int()
    m2060 = wc.eq(20).Or(wc.eq(60))
    water = water.where(m2060, c2060)

    # Remaining classes
    for code in [10, 30, 40, 50, 80, 90, 95]:
        pair = thr_map.get(code, fallbacks)
        c = cond_pair(pair).int()
        m = wc.eq(code)
        water = water.where(m, c)

    return water.selfMask()

# ---- Build water mask (S1: LULC-optimal thresholds / S2/L8: threshold 0) ----
if satellite == 'Sentinel-1':
    # Read threshold table here (base_dir set to '/your/local/directory')
    thr_map = _read_threshold_table("/your/local/directory", selected_layer, year, month)
    _assert_thresholds_complete(thr_map, selected_layer)

    if selected_layer in COMPARATOR_MAP:
        band, comp = COMPARATOR_MAP[selected_layer]
        water_mask = _apply_single_threshold_per_lulc(
            image, worldcover, band, comp, thr_map, FALLBACKS[selected_layer]
        )
        valid_mask = image.select(band).mask()
    elif selected_layer in ['VV_AND_VH', 'VV_OR_VH']:
        op = 'AND' if selected_layer == 'VV_AND_VH' else 'OR'
        water_mask = _apply_dual_threshold_per_lulc(
            image, worldcover, op, thr_map, FALLBACKS[selected_layer]
        )
        valid_mask = image.select('VV_DB').mask().And(image.select('VH_DB').mask())
    else:
        raise ValueError("For S1, selected_layer must be 'VV','VH','VVVH_RATIO','FOURVH_OVER_SUM','VV_AND_VH','VV_OR_VH'")

    # For country view, set NoData=white; otherwise, use dark gray
    nodata_color = '#FFFFFF' if REGION_KEY == 'country' else '#666666'

    # Create visualization image within the S1 branch
    vis_image = make_three_class_vis_from_masks(
        water_mask, valid_mask,
        water_col='#0000FF',
        non_col='#f5f5f5',  # #d9d9d9
        nodata_col=nodata_color
    ).clip(export_region)  # Do not use .unmask(0)

elif satellite in ['Sentinel-2', 'Landsat-8']:
    if selected_layer not in ['NDWI', 'MNDWI', 'WI2015']:
        raise ValueError("For optical, use 'NDWI','MNDWI','WI2015'.")
    band = selected_layer
    # Water where index > 0
    water_mask = image.select(band).gt(0)
    # Valid pixels of the selected band
    valid_mask = image.select(band).mask()

    vis_image = (
        make_three_class_vis_from_masks(water_mask, valid_mask)
        .clip(export_region)
        .unmask(0)
    )
else:
    raise ValueError("Satellite must be 'Sentinel-1','Sentinel-2', or 'Landsat-8'.")

ndwi_numpy = geemap.ee_to_numpy(vis_image, region=export_region, scale=scale)

if ndwi_numpy.ndim == 3:
    img = ndwi_numpy.astype(np.float32)
    if img.max() > 1.0:
        img /= 255.0
    # Transparency: true when all three bands are NaN or all zero
    mask_nan  = np.isnan(img).all(axis=2)
    mask_zero = (img[...,0]==0) & (img[...,1]==0) & (img[...,2]==0)
    alpha = (~(mask_nan | mask_zero)).astype(np.float32)

    img = np.nan_to_num(img, nan=0.0)
    rgba = np.dstack([img, alpha])
else:
    arr = ndwi_numpy.astype(np.float32)
    mask_nan  = np.isnan(arr)
    mask_zero = (arr==0)
    alpha = (~(mask_nan | mask_zero)).astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0)
    rgba = np.dstack([arr, arr, arr, alpha])

# ========= View Settings =========
bounds = export_region.bounds().getInfo()['coordinates'][0]
lon_min, lat_min = bounds[0]; lon_max, lat_max = bounds[2]

# Add a 0.1° margin for display only
pad = 0.1 if REGION_KEY == 'country' else 0.0

view_lon_min = lon_min - pad
view_lon_max = lon_max + pad
view_lat_min = lat_min - pad
view_lat_max = lat_max + pad

fig, ax = plt.subplots(figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_facecolor('white'); fig.patch.set_facecolor('white')

# Draw the image in the original extent (no stretching)
ax.imshow(
    rgba, origin='upper',
    extent=[lon_min, lon_max, lat_min, lat_max],
    transform=ccrs.PlateCarree(), interpolation='nearest', zorder=1
)

# Only the viewport gets the margin
ax.set_extent([view_lon_min, view_lon_max, view_lat_min, view_lat_max], crs=ccrs.PlateCarree())
ax.set_xlim(view_lon_min, view_lon_max)
ax.set_ylim(view_lat_min, view_lat_max)
ax.margins(0)

# Do not use set_boundary for country (useful only for small areas)
use_set_boundary = (REGION_KEY != 'country')
if use_set_boundary:
    from cartopy.mpl.patch import geos_to_path
    import matplotlib.path as mpath, numpy as _np
    region_fc = ee.FeatureCollection([ee.Feature(export_region)])
    region_geojson = geemap.ee_to_geojson(region_fc)
    region_geoms = [shape(f["geometry"]) for f in region_geojson["features"]]
    paths = list(geos_to_path(region_geoms))
    verts = _np.vstack([p.vertices for p in paths]); codes = _np.concatenate([p.codes for p in paths])
    region_path = mpath.Path(verts, codes)
    ax.set_boundary(region_path, transform=ccrs.PlateCarree())

# Frame
if 'geo' in ax.spines:
    ax.spines['geo'].set_visible(True)
    ax.spines['geo'].set_edgecolor('black')
    ax.spines['geo'].set_linewidth(1.2)
else:
    from matplotlib.patches import Rectangle
    frame = Rectangle((view_lon_min, view_lat_min),
                      view_lon_max - view_lon_min, view_lat_max - view_lat_min,
                      transform=ccrs.PlateCarree(), fill=False,
                      edgecolor='black', linewidth=1.2, zorder=9)
    ax.add_patch(frame)

plt.subplots_adjust(left=0, right=1, bottom=0, top=1)

# National boundaries (aligned precisely)
gaul_for_plot = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(export_region)
gaul_geojson = geemap.ee_to_geojson(gaul_for_plot)
gaul_geoms = [shape(f["geometry"]) for f in gaul_geojson["features"]]
faoborders = ShapelyFeature(gaul_geoms, ccrs.PlateCarree())
ax.add_feature(faoborders, edgecolor='black', facecolor='none', linewidth=0.3, zorder=10)

# Ticks: use integer degrees for 'country', otherwise keep existing style
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
if REGION_KEY == 'country':
    xticks = np.arange(np.ceil(view_lon_min), np.floor(view_lon_max) + 1, 1, dtype=int)
    yticks = np.arange(np.ceil(view_lat_min), np.floor(view_lat_max) + 1, 1, dtype=int)
    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format='.0f'))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format='.0f'))
else:
    nx, ny = 6, 6
    ax.set_xticks(np.linspace(view_lon_min, view_lon_max, nx), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(view_lat_min, view_lat_max, ny), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format='.2f'))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format='.2f'))

# Times New Roman & show ticks only on left/bottom
for t in ax.get_xticklabels() + ax.get_yticklabels():
    t.set_fontname('Times New Roman'); t.set_fontsize(14)
ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True, labeltop=False, direction='out')
ax.tick_params(axis='y', which='both', left=True, right=False, labelleft=True, labelright=False, direction='out')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# === Scale bar ===
def add_scalebar(ax, length_km, location='lower right', linewidth=3):
    degree_length = length_km / 111.32  # Approx. conversion along latitude
    if location == 'lower left':
        start_lon = view_lon_min + 0.10*(view_lon_max - view_lon_min)
        start_lat = view_lat_min + 0.01*(view_lat_max - view_lat_min)
    elif location == 'lower right':
        start_lon = view_lon_max - 0.10*(view_lon_max - view_lon_min) - degree_length
        start_lat = view_lat_min + 0.01*(view_lat_max - view_lat_min)
    else:
        raise ValueError("location must be 'lower left' or 'lower right'")

    end_lon = start_lon + degree_length
    ax.plot([start_lon, end_lon], [start_lat, start_lat],
            transform=ccrs.PlateCarree(), color="black", linewidth=linewidth)
    ax.text((start_lon + end_lon)/2, start_lat + 0.005*(view_lat_max - view_lat_min),
            f"{length_km} km", fontsize=16, color="black", ha="center", va="bottom",
            fontname='Times New Roman', transform=ccrs.PlateCarree())

# Example placement: lower right, 100 km
add_scalebar(ax, length_km=length_scalebar, location='lower right')

# ========= Save Figure =========
sat_abbr = {'Sentinel-1':'S1','Sentinel-2':'S2','Landsat-8':'L8'}[satellite]
out_dir = "GEE_Exports_Bangladesh"; os.makedirs(out_dir, exist_ok=True)
png_name = f"{sat_abbr}_{selected_layer}_{year}_{month:02d}_{region_label}.png"
png_path = os.path.join(out_dir, png_name)
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Image saved: {png_path}")

# ========= Vectorize Water =========
vectors = water_mask.selfMask().reduceToVectors(
    geometry=export_region, scale=scale, geometryType='polygon',
    eightConnected=True, labelProperty='water', reducer=ee.Reducer.countEvery()
)
shp_name = f"{sat_abbr}_{selected_layer}_{year}_{month:02d}_{region_label}_water.shp"
shp_path = os.path.join(out_dir, shp_name)
# geemap.ee_to_shp(vectors, filename=shp_path)  # Enable when needed
print(f"Shapefile saved: {shp_path}")
